# citrus-scout: training on Colab

This notebook is deliberately thin. All the logic lives in the repository, version
controlled and tested; the notebook only clones it, installs dependencies, and calls
the same CLI you would run locally.

That is the point: a result produced here is reproducible on any machine, because
nothing that affects it is defined in a notebook cell.

**Before running:** set the runtime to a GPU. Runtime > Change runtime type > T4 GPU.

**Expected time:** about 15 to 25 minutes for the 15-epoch baseline on a T4.

## 1. Check the GPU

If this reports no GPU, stop and change the runtime type. Training on a Colab CPU
would take hours.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv,noheader || echo "NO GPU: set Runtime > Change runtime type > T4 GPU"

## 2. Clone the repository

The repo is private, so cloning needs a token. Create a fine-grained personal access
token with read-only Contents permission on this one repository:
https://github.com/settings/tokens

Paste it when prompted. `getpass` keeps it out of the notebook output, so the token
does not end up saved in the file or in your Drive.

In [ ]:
import getpass
import os
import subprocess
from pathlib import Path

REPO = "allepuzz/citrus-scout"

if not Path("/content/citrus-scout").exists():
    token = getpass.getpass("GitHub token (input hidden): ")
    # Passed via argv rather than a shell string so the token never reaches
    # the shell history or a traceback.
    subprocess.run(
        ["git", "clone", f"https://{token}@github.com/{REPO}.git", "/content/citrus-scout"],
        check=True,
        capture_output=True,
    )
    del token
    print("cloned")
else:
    subprocess.run(["git", "-C", "/content/citrus-scout", "pull", "--quiet"], check=False)
    print("already present, pulled latest")

os.chdir("/content/citrus-scout")
print(subprocess.run(["git", "log", "--oneline", "-1"], capture_output=True, text=True).stdout)

## 3. Install

Colab already ships torch with CUDA, so we install the project's other dependencies
and the package itself, leaving the preinstalled torch alone. Reinstalling it would
cost several minutes and gain nothing.

In [ ]:
%pip install -q timm albumentations pydantic typer rich tqdm wandb scikit-learn
%pip install -q --no-deps -e .

import torch

print(f"torch {torch.__version__}, cuda available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"gpu: {torch.cuda.get_device_name(0)}")

## 4. Get the data

Two options. **Upload the archive** is the recommended one: run
`uv run citrus-scout data package` locally, put the resulting
`leaf_dataset.zip` in your Drive, and point this cell at it.

The archive carries its own split assignment, so training here uses exactly the
partition your local machine produced. Downloading from Kaggle instead would work,
but it re-derives the splits and pulls 5.5 GB every session.

In [ ]:
from pathlib import Path

# Option A: archive from Drive (recommended)
USE_DRIVE = True
DRIVE_ARCHIVE = "/content/drive/MyDrive/citrus-scout/leaf_dataset.zip"

archive = None

if USE_DRIVE:
    from google.colab import drive

    drive.mount("/content/drive")
    archive = Path(DRIVE_ARCHIVE)
    if not archive.exists():
        raise FileNotFoundError(
            f"{archive} not found.\n\n"
            "Locally, run:  uv run citrus-scout data package\n"
            "then upload data/processed/leaf_dataset.zip to that Drive path."
        )
    print(f"archive: {archive} ({archive.stat().st_size / 1e6:.0f} MB)")
else:
    # Option B: pull from Kaggle. Needs credentials uploaded to /root/.kaggle/
    !citrus-scout data download
    print("downloaded from Kaggle; splits will be rebuilt here")

## 5. Optional: Weights & Biases

Skip this cell if you would rather not log anywhere. Training prints metrics to the
output regardless, and writes `history.json` alongside the checkpoint.

In [ ]:
USE_WANDB = False

if USE_WANDB:
    import wandb

    wandb.login()

## 6. Smoke test

Two epochs to prove the pipeline runs before committing to a long job. Worth the two
minutes: a crash at epoch 12 of a 15-epoch run wastes far more.

In [ ]:
cmd = "citrus-scout train --config configs/leaf_smoke.yaml"
if archive:
    cmd += f" --archive {archive}"

!{cmd}

## 7. Train the baseline

A T4 has 16 GB, four times the laptop card this was written on, so the batch size is
raised accordingly. Larger batches also make the class weighting behave more
predictably, since each batch is more likely to contain minority-class examples.

In [ ]:
cmd = "citrus-scout train --config configs/leaf_baseline.yaml --batch-size 64"
if archive:
    cmd += f" --archive {archive}"
if USE_WANDB:
    cmd += " --wandb-project citrus-scout"

!{cmd}

## 8. Evaluate on the test split

Read the PPV column, not PR-AUC.

This test set is about 80% affected, because that is how the public datasets were
assembled. A real grove runs 2 to 5%. The evaluation projects the measured
sensitivity and specificity onto that real prevalence, and the last column tells you
how many trees a technician would visit per genuine case found.

That number, not the headline metric, is what decides whether the product is usable.

In [ ]:
cmd = "citrus-scout evaluate --checkpoint runs/leaf_baseline/best.pt"
if archive:
    cmd += f" --archive {archive}"

!{cmd}

## 9. Save the checkpoint

Colab wipes local storage when the session ends, so copy anything worth keeping to
Drive before closing the tab.

In [ ]:
import shutil

if USE_DRIVE:
    destination = Path("/content/drive/MyDrive/citrus-scout/runs")
    destination.mkdir(parents=True, exist_ok=True)

    for run in Path("runs").iterdir():
        if run.is_dir():
            shutil.copytree(run, destination / run.name, dirs_exist_ok=True)
            print(f"saved {run.name} to Drive")
else:
    from google.colab import files

    files.download("runs/leaf_baseline/best.pt")

## What these numbers do and do not mean

**They validate the pipeline.** Data loading, splitting without leakage, class
weighting, training, and honest evaluation all work end to end.

**They do not validate the product.** The training data is close-range leaf
photography against controlled backgrounds, mostly of diseases that do not occur in
Spain, with community-supplied labels that no agronomist or qPCR ever confirmed.

A high PR-AUC here says the model separates these particular photographs. It says
nothing yet about a drone over a Murcian grove, which is a different distribution in
every respect that matters: viewpoint, scale, lighting, background, and which
diseases are actually present.

What this earns is a working backbone and a pipeline ready for real imagery.